In [ ]:
import os, random, time, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# ── 1. CONFIGURATION & PATHS ──────────────────────────────────
SAVE_DIR_IMG = Path('/content/drive/MyDrive/Model_results')
SAVE_DIR_IMG.mkdir(parents=True, exist_ok=True)

# TODO: Define your local or drive paths for images and metadata
IMAGE_DIR       = Path('path/to/images')
METADATA_CSV    = Path('path/to/metadata.csv')
GROUNDTRUTH_CSV = Path('path/to/groundtruth.csv')

# TODO: Experiment with these hyperparameters to optimize your model
CFG = dict(
    img_size      = 224,
    batch_size    = ...,   # Suggested: 16, 32, or 64
    num_workers   = 2,
    num_epochs    = ...,   # How many passes through the data?
    lr            = ...,   # Common starting points: 1e-4 or 5e-5
    weight_decay  = 1e-4,
    dropout       = ...,   # Help prevent overfitting (e.g., 0.3 - 0.5)
    test_split    = 0.2,
    seed          = 42,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(CFG['seed'])

# ── 2. DATA PREPARATION (BINARY IMAGE-ONLY) ───────────────────
meta = pd.read_csv(METADATA_CSV)
gt   = pd.read_csv(GROUNDTRUTH_CSV)

# Define which classes are considered Malignant for binary classification
MALIGNANT_CLASSES = ['AKIEC', 'BCC', 'MAL_OTH', 'MEL', 'SCCKA']
ORIG_CLASS_COLS = ['AKIEC', 'BCC', 'BEN_OTH', 'BKL', 'DF', 'INF', 'MAL_OTH', 'MEL', 'NV', 'SCCKA', 'VASC']

# Create binary labels (1 for Malignant, 0 for Benign)
gt['label'] = gt[ORIG_CLASS_COLS].idxmax(axis=1).apply(lambda x: 1 if x in MALIGNANT_CLASSES else 0)
df = meta.merge(gt[['lesion_id', 'label']], on='lesion_id', how='inner')

# Split data by lesion_id to prevent data leakage (images from same lesion in both train/test)
unique_lesions = df.drop_duplicates(subset='lesion_id')
train_l, temp_l = train_test_split(unique_lesions['lesion_id'], test_size=CFG['test_split'],
                                  stratify=unique_lesions['label'], random_state=CFG['seed'])
val_l, test_l = train_test_split(temp_l, test_size=0.50,
                                stratify=unique_lesions[unique_lesions['lesion_id'].isin(temp_l)]['label'],
                                random_state=CFG['seed'])

train_df = df[df['lesion_id'].isin(train_l)].reset_index(drop=True)
val_df   = df[df['lesion_id'].isin(val_l)].reset_index(drop=True)
test_df  = df[df['lesion_id'].isin(test_l)].reset_index(drop=True)

# ── 3. DATASET & TRANSFORMATIONS ─────────────────────────────
class ISICDatasetImage(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df, self.image_dir, self.transform = dataframe, Path(image_dir), transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = None
        # Handle various image extensions
        for ext in ('.jpg', '.jpeg', '.png', '.JPG'):
            p = self.image_dir / str(row['lesion_id']) / f"{row['isic_id']}{ext}"
            if p.exists():
                img = Image.open(p).convert('RGB')
                break
        if img is None: img = Image.new('RGB', (CFG['img_size'], CFG['img_size']))
        if self.transform: img = self.transform(img)
        return img, int(row['label'])

# TODO: Research Data Augmentation. What transforms help a model generalize?
train_tfm = transforms.Compose([
    transforms.Resize((CFG['img_size']+32, CFG['img_size']+32)),
    transforms.RandomCrop(CFG['img_size']),
    # Add more augmentations here (e.g., flips, rotations, color jitter)
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_tfm = transforms.Compose([
    transforms.Resize((CFG['img_size'], CFG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ── 4. MODEL ─────────────────────────────────────────────────
class ISICModelImage(nn.Module):
    def __init__(self, num_classes=2, dropout=0.4):
        super().__init__()
        # TODO: Select a backbone model from `timm` (e.g., 'efficientnet_b0', 'resnet34', or 'mobilenetv3_large_100')
        self.backbone = timm.create_model('...', pretrained=True, num_classes=0)
        img_dim = self.backbone.num_features

        # TODO: Design your classification head
        self.head = nn.Sequential(
            nn.Linear(img_dim, ...),
            nn.BatchNorm1d(...),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(..., num_classes)
        )

    def forward(self, x):
        x = self.backbone.forward_features(x)
        x = nn.AdaptiveAvgPool2d(1)(x).flatten(1)
        return self.head(x)

# ── 5. TRAINING HELPER ───────────────────────────────────────
def run_epoch_img(model, loader, criterion, optimizer=None, phase='train'):
    model.train() if phase == 'train' else model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_labels, all_probs = [], []

    with torch.set_grad_enabled(phase == 'train'):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss = criterion(logits, labels)

            if phase == 'train' and optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            probs = torch.softmax(logits, dim=1)
            running_loss += loss.item() * imgs.size(0)
            correct += (probs.argmax(dim=1) == labels).sum().item()
            total += imgs.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

    # Calculate AUC for the positive class (Malignant)
    auc = roc_auc_score(all_labels, np.array(all_probs)[:, 1])
    return running_loss/total, correct/total, auc, all_labels, all_probs

# ── 6. EXECUTION ──────────────────────────────────────────────

# TODO: Implement DataLoaders.
# HINT: Use WeightedRandomSampler to address class imbalance between Benign and Malignant samples.
train_loader = DataLoader(...)
val_loader = DataLoader(...)
test_loader = DataLoader(...)

model = ISICModelImage(dropout=CFG['dropout']).to(DEVICE)

# TODO: Initialize your optimizer and scheduler
optimizer = optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = ... # e.g., optim.lr_scheduler.CosineAnnealingLR or ReduceLROnPlateau
criterion = nn.CrossEntropyLoss(...)

history = {'epoch':[], 'tr_auc':[], 'vl_auc':[], 'tr_loss':[], 'vl_loss':[], 'tr_acc':[], 'vl_acc':[]}
best_val_auc = 0.0

for epoch in range(1, CFG['num_epochs'] + 1):
    t0 = time.time()
    tr_loss, tr_acc, tr_auc, _, _ = run_epoch_img(model, train_loader, criterion, optimizer, 'train')
    vl_loss, vl_acc, vl_auc, _, _ = run_epoch_img(model, val_loader, criterion, None, 'val')

    # scheduler.step() logic here

    for k, v in zip(history.keys(), [epoch, tr_auc, vl_auc, tr_loss, vl_loss, tr_acc, vl_acc]):
        history[k].append(v)

    print(f"--- Epoch {epoch:02d} Summary ---")
    print(f"TRAIN | Loss: {tr_loss:.4f} | Acc: {tr_acc:.4f} | AUC: {tr_auc:.4f}")
    print(f"VAL   | Loss: {vl_loss:.4f} | Acc: {vl_acc:.4f} | AUC: {vl_auc:.4f}")
    print(f"Time  | {time.time()-t0:.1f}s")

    # Save the best model
    if vl_auc > best_val_auc:
        best_val_auc = vl_auc
        print(f"✅ Saving best model based on Val AUC: {best_val_auc:.4f}\n")
        torch.save(model.state_dict(), SAVE_DIR_IMG / 'best_image_model.pth')
    else:
        print("\n")

# ── 7. FINAL EVALUATION & PLOTTING ────────────────────────────
print('\n--- Final Evaluation (Image-Only) ---')

# TODO: Load the saved best weights and evaluate on the unseen test_loader
model.load_state_dict(torch.load(...))
ts_loss, ts_acc, ts_auc, ts_labels, ts_probs = run_epoch_img(model, test_loader, criterion, phase='test')

ts_preds = np.array(ts_probs).argmax(axis=1)
print(f'Test AUC: {ts_auc:.4f} | Test Acc: {ts_acc:.4f}')
print(classification_report(ts_labels, ts_preds, target_names=['Benign', 'Malignant']))

# Plotting boilerplate remains to help students visualize results
cm = confusion_matrix(ts_labels, ts_preds)
plt.figure(figsize=(6,5)); sns.heatmap(cm, annot=True, fmt='d', cmap='Greens'); plt.title('Test CM'); plt.show()

epochs = history['epoch']
plt.figure(figsize=(15,5))
plt.subplot(1,3,1); plt.plot(epochs, history['tr_auc'], label='Train'); plt.plot(epochs, history['vl_auc'], label='Val'); plt.title('AUC'); plt.legend()
plt.subplot(1,3,2); plt.plot(epochs, history['tr_loss'], label='Train'); plt.plot(epochs, history['vl_loss'], label='Val'); plt.title('Loss'); plt.legend()
plt.subplot(1,3,3); plt.plot(epochs, history['tr_acc'], label='Train'); plt.plot(epochs, history['vl_acc'], label='Val'); plt.title('Accuracy'); plt.legend()
plt.tight_layout(); plt.savefig(SAVE_DIR_IMG / 'binary_image_plots.png'); plt.show()